# NiyamTrace-X — T4-Optimized Local External Benchmark Closure

This version is specifically designed for a **Google Colab NVIDIA T4 (15 GiB)**.

It does **not** use OpenAI, Gemini, Groq, or OpenRouter APIs.

## T4 changes

- vLLM remains isolated in its own Python 3.12 environment.
- Every model is served in **FP16**.
- `--enforce-eager` reduces CUDA-graph memory overhead.
- GPU allocation is capped at **78%**.
- Context length is capped at **4096** in CLOSURE mode.
- Maximum concurrent sequences are capped at **2**.
- Every model and benchmark stage is checkpointed.
- The old long Cell 7 is replaced by **one model per cell**.
- If a cell/runtime stops, rerunning reuses completed results that remain on disk.

## Core model families for the paper closure gate

1. Granite — `ibm-granite/granite-3.1-2b-instruct`
2. Qwen — `Qwen/Qwen2.5-3B-Instruct`
3. Phi — `microsoft/Phi-4-mini-instruct`
4. Mistral — `mistralai/Ministral-3-3B-Instruct-2512`

The closure gate needs **any three independent core families** to complete BFCL-v4, AgentDojo, and tau3.

## Additional T4 external models

- `Salesforce/xLAM-1B-fc-r`
- `Salesforce/xLAM-3B-fc-r`
- `Qwen/Qwen2.5-7B-Instruct-AWQ`
- optional gated `meta-llama/Llama-3.2-3B-Instruct`

Supplementary models broaden the external evidence but are not required for closure.

At the end download `NTX_T4_LOCAL_EXTERNAL_CLOSURE_RESULTS.zip`.


In [ ]:
# CELL 1 — T4-SPECIFIC CONFIGURATION
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, re, time, random, hashlib, zipfile, shutil, subprocess, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED)
np.random.seed(SEED)

MODE=os.getenv("NTX_T4_MODE","CLOSURE").upper()
assert MODE in {"SMOKE","CLOSURE","FULL"}

BASE=Path("/content/NTX_T4_EXTERNAL_CLOSURE")
WORK=BASE/"work"
MODELS=BASE/"models"
RAW=BASE/"raw"
RESULTS=BASE/"results"
LOGS=BASE/"logs"
PAPER=BASE/"paper_integration"
ARCH=BASE/"archives"
TEMPLATES=BASE/"templates"
for p in [BASE,WORK,MODELS,RAW,RESULTS,LOGS,PAPER,ARCH,TEMPLATES]:
    p.mkdir(parents=True,exist_ok=True)

CFG={
    "SMOKE":{
        "bfcl_limit":3,
        "dojo_suites":["banking"],
        "dojo_user_tasks":["user_task_0"],
        "dojo_injection_tasks":["injection_task_0"],
        "tau_domains":["airline"],
        "tau_tasks":1,
        "tau_steps":14,
        "max_model_len":3072,
        "gpu_memory_utilization":0.74,
    },
    "CLOSURE":{
        "bfcl_limit":12,
        "dojo_suites":["banking","workspace","travel","slack"],
        "dojo_user_tasks":["user_task_0"],
        "dojo_injection_tasks":["injection_task_0"],
        "tau_domains":["airline","retail","telecom"],
        "tau_tasks":1,
        "tau_steps":26,
        "max_model_len":4096,
        "gpu_memory_utilization":0.78,
    },
    "FULL":{
        "bfcl_limit":50,
        "dojo_suites":["banking","workspace","travel","slack"],
        "dojo_user_tasks":["user_task_0","user_task_5"],
        "dojo_injection_tasks":["injection_task_0","injection_task_1"],
        "tau_domains":["airline","retail","telecom"],
        "tau_tasks":3,
        "tau_steps":40,
        "max_model_len":6144,
        "gpu_memory_utilization":0.80,
    }
}[MODE]

PORT=8000
LOCAL_BASE=f"http://127.0.0.1:{PORT}/v1"
LOCAL_KEY="EMPTY"
KEEP_MODEL_WEIGHTS=True
HF_TOKEN=os.getenv("HF_TOKEN","").strip()
CHECKPOINT=RESULTS/"T4_CHECKPOINT.json"

print("MODE:",MODE)
print(json.dumps(CFG,indent=2))


In [ ]:
# CELL 2 — INSTALL LOCAL INFERENCE + UTILITIES (FIXED)
# Root-cause fix:
# vLLM is installed in a clean Python 3.12 uv environment instead of
# Colab's base Python environment. This prevents stale Colab TorchAudio/
# TorchVision packages from being imported against a different CUDA build.

def sh(cmd,cwd=None,env=None,timeout=None):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        capture_output=True,
        text=True,
        errors="replace",
        timeout=timeout,
    )

def save_log(name,p,cmd=None):
    text=""
    if cmd:
        text+="COMMAND\n"+" ".join(map(str,cmd))+"\n\n"
    text+="STDOUT\n"+(p.stdout or "")+"\n\nSTDERR\n"+(p.stderr or "")
    (LOGS/name).write_text(text,errors="ignore")

def ensure_base_package(import_name,pip_spec=None):
    try:
        __import__(import_name)
        return
    except Exception:
        pass
    spec=pip_spec or import_name
    p=sh([sys.executable,"-m","pip","install","-q","-U",spec])
    save_log(f"install_{import_name}.log",p)
    if p.returncode:
        print(p.stderr[-4000:])
        raise RuntimeError(f"Could not install {spec}")

# uv first.
if shutil.which("uv") is None:
    p=sh([sys.executable,"-m","pip","install","-q","-U","uv"])
    save_log("install_uv.log",p)
    if p.returncode:
        raise RuntimeError("uv installation failed")

# Lightweight clients remain in base runtime.
ensure_base_package("openai","openai")
ensure_base_package("huggingface_hub","huggingface_hub")
ensure_base_package("psutil","psutil")

from openai import OpenAI
from huggingface_hub import snapshot_download, HfApi

# ------------------------------------------------------------
# Isolated vLLM environment
# ------------------------------------------------------------

VLLM_ENV=WORK/"vllm_py312"
VLLM_PY=VLLM_ENV/"bin"/"python"

# Managed Python 3.12 is the stable isolated runtime.
p=sh(["uv","python","install","3.12"])
save_log("vllm_python312_install.log",p)

if not VLLM_ENV.exists():
    p=sh([
        "uv","venv",VLLM_ENV,
        "--python","3.12",
        "--seed",
        "--managed-python",
    ])
    save_log("vllm_venv_create.log",p)
    if p.returncode:
        raise RuntimeError("Could not create isolated vLLM Python 3.12 environment.")

# Install vLLM with uv selecting a CUDA/PyTorch backend compatible with
# the runtime driver. This does NOT mutate Colab's base torch/torchaudio.
p=sh([
    "uv","pip","install",
    "--python",VLLM_PY,
    "-U",
    "vllm",
    "--torch-backend=auto",
])
save_log("vllm_isolated_install.log",p)
if p.returncode:
    print(p.stderr[-6000:])
    raise RuntimeError("Isolated vLLM installation failed.")

# Critical verification: vLLM + torch import in the exact interpreter
# that will launch the API server. Also confirm a stale torchaudio package
# is not visible in this isolated environment.
verify_code = r"""
import importlib.util, json, torch, vllm
print(json.dumps({
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "vllm": vllm.__version__,
    "torchaudio_visible": importlib.util.find_spec("torchaudio") is not None,
    "torchvision_visible": importlib.util.find_spec("torchvision") is not None,
}))
"""

v=sh([VLLM_PY,"-c",verify_code])
save_log("vllm_isolated_verify.log",v,[VLLM_PY,"-c","<verification>"])
if v.returncode:
    print(v.stderr[-6000:])
    raise RuntimeError("vLLM isolated runtime verification failed.")

print("Isolated vLLM runtime:")
print(v.stdout.strip())

# Record environment provenance.
fr=sh(["uv","pip","freeze","--python",VLLM_PY])
(LOGS/"vllm_isolated_freeze.txt").write_text(fr.stdout or "")

print("✅ Isolated vLLM environment is ready for the T4.")


In [ ]:
# CELL 2B — FAIL-FAST LOCAL vLLM RUNTIME DIAGNOSTIC
# This must pass BEFORE any model download begins.

diag = sh([
    VLLM_PY,
    "-c",
    (
        "import torch,vllm,importlib.util;"
        "print('TORCH',torch.__version__);"
        "print('CUDA',torch.version.cuda);"
        "print('VLLM',vllm.__version__);"
        "print('TORCHAUDIO_VISIBLE',importlib.util.find_spec('torchaudio') is not None)"
    ),
])

print(diag.stdout)
if diag.returncode:
    print(diag.stderr)
    raise RuntimeError("Isolated vLLM runtime diagnostic failed.")

if "TORCH " not in diag.stdout or "VLLM " not in diag.stdout:
    raise RuntimeError("Unexpected vLLM diagnostic output.")

print("✅ Runtime diagnostic passed. Model downloads may proceed.")


In [ ]:
# CELL 3 — VERIFY T4 + DEFINE T4-SAFE MODEL QUEUE

def gpu_info():
    p=sh([
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader,nounits",
    ])
    if p.returncode:
        raise RuntimeError("No NVIDIA GPU found. Select a GPU runtime in Colab.")
    line=p.stdout.strip().splitlines()[0]
    name,mem=line.rsplit(",",1)
    return name.strip(),float(mem.strip())/1024

GPU_NAME,VRAM_GB=gpu_info()
DISK_FREE_GB=shutil.disk_usage("/content").free/(1024**3)

print("GPU:",GPU_NAME)
print(f"VRAM: {VRAM_GB:.1f} GiB")
print(f"Disk free: {DISK_FREE_GB:.1f} GiB")

if "T4" not in GPU_NAME.upper():
    print("⚠️ Tuned for T4; continuing on:",GPU_NAME)

MODEL_REGISTRY=[
    {
        "slug":"granite31_2b",
        "family":"Granite",
        "repo":"ibm-granite/granite-3.1-2b-instruct",
        "tool_parser":"granite",
        "min_vram_gb":6.5,
        "full_bench":True,
        "counts_for_closure":True,
        "gated":False,
        "extra_server_args":[],
    },
    {
        "slug":"qwen25_3b",
        "family":"Qwen",
        "repo":"Qwen/Qwen2.5-3B-Instruct",
        "tool_parser":"hermes",
        "min_vram_gb":7.5,
        "full_bench":True,
        "counts_for_closure":True,
        "gated":False,
        "extra_server_args":[],
    },
    {
        "slug":"phi4_mini",
        "family":"Phi",
        "repo":"microsoft/Phi-4-mini-instruct",
        "tool_parser":"phi4_mini_json",
        "min_vram_gb":9.0,
        "full_bench":True,
        "counts_for_closure":True,
        "gated":False,
        "chat_template_key":"phi4",
        "extra_server_args":["--trust-remote-code"],
    },
    {
        "slug":"ministral3_3b",
        "family":"Mistral",
        "repo":"mistralai/Ministral-3-3B-Instruct-2512",
        "tool_parser":"mistral",
        "min_vram_gb":9.5,
        "full_bench":True,
        "counts_for_closure":True,
        "gated":False,
        "extra_server_args":["--language-model-only"],
    },
    {
        "slug":"xlam_1b",
        "family":"xLAM",
        "repo":"Salesforce/xLAM-1B-fc-r",
        "tool_parser":"xlam",
        "min_vram_gb":5.5,
        "full_bench":False,
        "counts_for_closure":False,
        "gated":False,
        "chat_template_key":"xlam_qwen",
        "extra_server_args":[],
    },
    {
        "slug":"xlam_3b",
        "family":"xLAM",
        "repo":"Salesforce/xLAM-3B-fc-r",
        "tool_parser":"xlam",
        "min_vram_gb":7.5,
        "full_bench":False,
        "counts_for_closure":False,
        "gated":False,
        "chat_template_key":"xlam_qwen",
        "extra_server_args":[],
    },
    {
        "slug":"qwen25_7b_awq",
        "family":"Qwen",
        "repo":"Qwen/Qwen2.5-7B-Instruct-AWQ",
        "tool_parser":"hermes",
        "min_vram_gb":10.5,
        "full_bench":False,
        "counts_for_closure":False,
        "gated":False,
        "extra_server_args":[],
    },
    {
        "slug":"llama32_3b",
        "family":"Llama",
        "repo":"meta-llama/Llama-3.2-3B-Instruct",
        "tool_parser":"llama3_json",
        "min_vram_gb":8.0,
        "full_bench":False,
        "counts_for_closure":True,
        "gated":True,
        "extra_server_args":[],
    },
]

for x in MODEL_REGISTRY:
    x["gpu_eligible"]=x["min_vram_gb"]<=VRAM_GB
    x["token_eligible"]=(not x["gated"]) or bool(HF_TOKEN)

registry_df=pd.DataFrame(MODEL_REGISTRY)
display(registry_df[[
    "slug","family","repo","min_vram_gb","gpu_eligible",
    "full_bench","counts_for_closure","gated","token_eligible"
]])
registry_df.to_csv(RESULTS/"00_t4_model_registry.csv",index=False)


In [ ]:
# CELL 4 — SET UP ALL THREE EXTERNAL BENCHMARKS ONCE

def clone_once(url,dest):
    dest=Path(dest)
    if not dest.exists():
        p=sh(["git","clone","--depth","1",url,dest])
        save_log("clone_"+dest.name+".log",p)
        if p.returncode:
            raise RuntimeError(f"Clone failed: {url}")
    return sh(["git","-C",dest,"rev-parse","HEAD"]).stdout.strip()

# ---------- BFCL / EvalScope isolated Python 3.11 ----------
BFENV=WORK/"bfcl_env"
sh(["uv","python","install","3.11"])
if not BFENV.exists():
    p=sh(["uv","venv",BFENV,"--python","3.11"])
    save_log("bfcl_venv.log",p)
    if p.returncode:
        raise RuntimeError("BFCL venv creation failed")
BFPY=BFENV/"bin"/"python"
p=sh(["uv","pip","install","--python",BFPY,"-U","evalscope[bfcl]"])
save_log("bfcl_install.log",p)
if p.returncode:
    raise RuntimeError("BFCL/EvalScope install failed")
v=sh([BFPY,"-c","from evalscope import run_task; from evalscope.config import TaskConfig; print('BFCL_READY')"])
if v.returncode or "BFCL_READY" not in v.stdout:
    raise RuntimeError("BFCL environment verification failed")

# ---------- AgentDojo ----------
DOJO=WORK/"agentdojo"
DOJO_COMMIT=clone_once("https://github.com/ethz-spylab/agentdojo.git",DOJO)
p=sh(["uv","sync"],cwd=DOJO)
save_log("dojo_sync.log",p)
if p.returncode:
    raise RuntimeError("AgentDojo uv sync failed")
h=sh(["uv","run","python","-m","agentdojo.scripts.benchmark","--help"],cwd=DOJO)
save_log("dojo_help.log",h)
ht=(h.stdout or "")+(h.stderr or "")
for token in ["openai-compatible","--model-id","--force-rerun"]:
    if token not in ht:
        raise RuntimeError(f"AgentDojo CLI missing {token}")

# ---------- tau2 / tau3 ----------
TAU=WORK/"tau2-bench"
TAU_COMMIT=clone_once("https://github.com/sierra-research/tau2-bench.git",TAU)
p=sh(["uv","sync"],cwd=TAU)
save_log("tau_sync.log",p)
if p.returncode:
    raise RuntimeError("tau2 uv sync failed")
TAUPY=TAU/".venv"/"bin"/"python"
p=sh(["uv","pip","install","--python",TAUPY,"websockets","soundfile"],cwd=TAU)
save_log("tau_extra_deps.log",p)
v=sh([TAUPY,"-c","import websockets,soundfile;print('TAU_READY')"],cwd=TAU)
if v.returncode or "TAU_READY" not in v.stdout:
    raise RuntimeError("tau2 dependency verification failed")

BENCHMARK_VERSIONS={
    "agentdojo_commit":DOJO_COMMIT,
    "tau2_commit":TAU_COMMIT,
}
(Path(RESULTS/"01_benchmark_versions.json")
 .write_text(json.dumps(BENCHMARK_VERSIONS,indent=2)))

print("BFCL, AgentDojo, and tau2 environments ready.")

# Extra CLI checks before spending GPU time.
for token in ["--suite","--user-task","--injection-task","--max-workers"]:
    if token not in ht:
        raise RuntimeError(f"AgentDojo CLI unexpectedly missing {token}")

tau_help=sh(["uv","run","tau2","run","--help"],cwd=TAU)
save_log("tau_run_help.log",tau_help)
tau_ht=(tau_help.stdout or "")+(tau_help.stderr or "")
for token in ["--num-tasks","--max-steps","--auto-resume","--agent-llm-args","--user-llm-args"]:
    if token not in tau_ht:
        raise RuntimeError(f"tau2 CLI unexpectedly missing {token}")


In [ ]:
# CELL 5 — T4 MODEL DOWNLOAD + LOCAL vLLM SERVER

from urllib.request import urlretrieve
from huggingface_hub import snapshot_download, HfApi

SERVER=None

TEMPLATE_URLS={
    "phi4":
        "https://raw.githubusercontent.com/vllm-project/vllm/main/"
        "examples/tool_chat_template_llama.jinja",
    "xlam_qwen":
        "https://raw.githubusercontent.com/vllm-project/vllm/main/"
        "examples/tool_chat_template_xlam_qwen.jinja",
}

def ensure_template(key):
    if not key:
        return None
    dest=TEMPLATES/f"{key}.jinja"
    if not dest.exists():
        print("Downloading tool template:",key)
        urlretrieve(TEMPLATE_URLS[key],dest)
    return dest

def load_checkpoint():
    try:
        return json.loads(CHECKPOINT.read_text())
    except Exception:
        return {}

def save_checkpoint(state):
    CHECKPOINT.write_text(json.dumps(state,indent=2,default=str))

STATE=load_checkpoint()

def remote_model_info(repo):
    api=HfApi(token=HF_TOKEN or None)
    info=api.model_info(repo,files_metadata=True)
    size=sum((getattr(s,"size",0) or 0) for s in info.siblings)/(1024**3)
    return info.sha,size

def download_model(spec):
    if spec.get("gated") and not HF_TOKEN:
        raise RuntimeError("GATED_MODEL_NO_HF_TOKEN")

    revision,size_gb=remote_model_info(spec["repo"])
    free=shutil.disk_usage("/content").free/(1024**3)
    required=max(3.0,size_gb*1.08+2.0)
    if free<required:
        raise RuntimeError(
            f"Not enough disk for {spec['repo']}: need ~{required:.1f} GiB, have {free:.1f}"
        )

    print(f"Downloading/reusing {spec['repo']} ({size_gb:.1f} GiB remote files)")
    path=snapshot_download(
        repo_id=spec["repo"],
        revision=revision,
        token=HF_TOKEN or None,
        cache_dir=str(MODELS/"hf_cache"),
    )
    return Path(path),revision,size_gb

def stop_server():
    global SERVER
    if SERVER is not None:
        try:
            SERVER.terminate()
            SERVER.wait(timeout=20)
        except Exception:
            try: SERVER.kill()
            except Exception: pass
        SERVER=None

    subprocess.run(
        ["pkill","-f","vllm.entrypoints.openai.api_server"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(4)
    gc.collect()

def start_server(spec,model_path):
    global SERVER
    stop_server()

    logfile=LOGS/f"server_{spec['slug']}.log"
    fh=open(logfile,"w")

    cmd=[
        VLLM_PY,
        "-m","vllm.entrypoints.openai.api_server",
        "--model",str(model_path),
        "--served-model-name",spec["slug"],
        "--host","127.0.0.1",
        "--port",str(PORT),
        "--dtype","half",
        "--gpu-memory-utilization",str(CFG["gpu_memory_utilization"]),
        "--max-model-len",str(CFG["max_model_len"]),
        "--max-num-seqs","2",
        "--swap-space","2",
        "--enforce-eager",
        "--enable-auto-tool-choice",
        "--tool-call-parser",spec["tool_parser"],
        "--disable-log-requests",
    ]

    template=ensure_template(spec.get("chat_template_key"))
    if template:
        cmd += ["--chat-template",str(template)]

    cmd += list(spec.get("extra_server_args",[]))

    print("Starting T4 vLLM:"," ".join(map(str,cmd)))
    SERVER=subprocess.Popen(
        [str(x) for x in cmd],
        stdout=fh,
        stderr=subprocess.STDOUT,
        cwd=str(BASE),
        env=os.environ.copy(),
    )

    client=OpenAI(api_key=LOCAL_KEY,base_url=LOCAL_BASE)
    deadline=time.time()+420
    last_error=""

    while time.time()<deadline:
        if SERVER.poll() is not None:
            fh.flush()
            tail=logfile.read_text(errors="ignore")[-9000:]
            raise RuntimeError("vLLM server exited during startup:\n"+tail)
        try:
            models=client.models.list()
            if models.data:
                print("Server ready:",models.data[0].id)
                return client
        except Exception as e:
            last_error=repr(e)
        time.sleep(5)

    stop_server()
    raise RuntimeError("Timed out waiting for vLLM server: "+last_error)

def tool_preflight(client,spec):
    result={
        "slug":spec["slug"],"family":spec["family"],"repo":spec["repo"],
        "chat_ok":False,"auto_tool_ok":False,"forced_tool_ok":False,"error":"",
    }

    tool=[{
        "type":"function",
        "function":{
            "name":"lookup_order",
            "description":"Look up an order by ID",
            "parameters":{
                "type":"object",
                "properties":{"order_id":{"type":"string"}},
                "required":["order_id"],
            },
        },
    }]

    try:
        r=client.chat.completions.create(
            model=spec["slug"],
            messages=[{"role":"user","content":"Reply exactly OK."}],
            temperature=0,max_tokens=32,
        )
        result["chat_ok"]=bool(r.choices)
    except Exception as e:
        result["error"]="CHAT:"+repr(e)
        return result

    try:
        r=client.chat.completions.create(
            model=spec["slug"],
            messages=[{
                "role":"user",
                "content":"Look up order A123 using the provided lookup_order tool before answering."
            }],
            tools=tool,
            tool_choice="auto",
            temperature=0,max_tokens=160,
        )
        calls=getattr(r.choices[0].message,"tool_calls",None) if r.choices else None
        result["auto_tool_ok"]=bool(calls and calls[0].function.name=="lookup_order")
    except Exception as e:
        result["error"]+=" AUTO:"+repr(e)

    try:
        r=client.chat.completions.create(
            model=spec["slug"],
            messages=[{"role":"user","content":"Call lookup_order for A123."}],
            tools=tool,
            tool_choice={"type":"function","function":{"name":"lookup_order"}},
            temperature=0,max_tokens=160,
        )
        calls=getattr(r.choices[0].message,"tool_calls",None) if r.choices else None
        result["forced_tool_ok"]=bool(calls and calls[0].function.name=="lookup_order")
    except Exception as e:
        result["error"]+=" FORCED:"+repr(e)

    return result

print("✅ T4 model lifecycle helpers ready.")


In [ ]:
# CELL 6 — RESUMABLE BENCHMARK RUNNERS

def safe_run(cmd,cwd=None,env=None,timeout=1800):
    try:
        return sh(cmd,cwd=cwd,env=env,timeout=timeout),None
    except subprocess.TimeoutExpired as e:
        class R:
            returncode=124
            stdout=(e.stdout or "") if isinstance(e.stdout,str) else ""
            stderr=((e.stderr or "") if isinstance(e.stderr,str) else "")+"\nTIMEOUT"
        return R(),"TIMEOUT"

def parse_bfcl(root,spec):
    rec=[]
    root=Path(root)
    for p in root.rglob("*"):
        if not p.is_file():continue
        rel=str(p.relative_to(root))
        try:
            if p.suffix.lower()==".csv":
                df=pd.read_csv(p)
                for c in df.columns:
                    if any(k in str(c).lower() for k in ["accuracy","score"]):
                        for v in pd.to_numeric(df[c],errors="coerce").dropna():
                            rec.append({
                                "benchmark":"BFCL-v4","slug":spec["slug"],
                                "family":spec["family"],"model":spec["repo"],
                                "slice":rel,"metric":str(c),"score":float(v),
                            })
            elif p.suffix.lower() in {".json",".jsonl"}:
                texts=(p.read_text(errors="ignore").splitlines()
                       if p.suffix.lower()==".jsonl" else [p.read_text(errors="ignore")])
                for txt in texts:
                    try:o=json.loads(txt)
                    except Exception:continue
                    stack=[("",o)]
                    while stack:
                        path,x=stack.pop()
                        if isinstance(x,dict):
                            for k,v in x.items():
                                q=f"{path}.{k}" if path else str(k)
                                if isinstance(v,(dict,list)):
                                    stack.append((q,v))
                                elif isinstance(v,(int,float)) and any(
                                    t in str(k).lower() for t in ["accuracy","score"]
                                ):
                                    rec.append({
                                        "benchmark":"BFCL-v4","slug":spec["slug"],
                                        "family":spec["family"],"model":spec["repo"],
                                        "slice":rel,"metric":q,"score":float(v),
                                    })
                        elif isinstance(x,list):
                            for i,v in enumerate(x):stack.append((f"{path}[{i}]",v))
        except Exception:
            pass
    return pd.DataFrame(rec).drop_duplicates() if rec else pd.DataFrame()

def run_bfcl(spec):
    out=RAW/"bfcl"/spec["slug"]
    out.mkdir(parents=True,exist_ok=True)
    result_file=RESULTS/f"bfcl_{spec['slug']}.csv"

    if result_file.exists() and result_file.stat().st_size>2:
        old=pd.read_csv(result_file)
        if len(old):
            return "SUPPORTED_REUSED",old,0

    runner=WORK/f"run_bfcl_{spec['slug']}.py"
    runner.write_text(f"""from evalscope import run_task
from evalscope.config import TaskConfig
cfg=TaskConfig(
 model={spec['slug']!r},
 api_url={LOCAL_BASE!r},
 api_key={LOCAL_KEY!r},
 eval_type='openai_api',
 datasets=['bfcl_v4'],
 work_dir={str(out)!r},
 limit={CFG['bfcl_limit']!r},
 seed={SEED},
 generation_config={{'temperature':0.0,'max_tokens':768,'timeout':120}},
 dataset_args={{'bfcl_v4':{{'extra_params':{{'is_fc_model':True}}}}}}
)
run_task(task_cfg=cfg)
""")

    p,timeout=safe_run([BFPY,runner],cwd=out,timeout=1800)
    save_log(f"bfcl_{spec['slug']}.log",p,[BFPY,runner])
    d=parse_bfcl(out,spec)
    d.to_csv(result_file,index=False)
    st="SUPPORTED" if len(d) else ("TIMEOUT" if timeout else "FAILED")
    return st,d,p.returncode

def parse_dojo(root,spec,suite):
    rec=[]
    for p in Path(root).rglob("*.json"):
        try:o=json.loads(p.read_text())
        except Exception:continue
        if not isinstance(o,dict):continue
        u=o.get("utility")
        s=o.get("security")
        if not isinstance(u,bool) and not isinstance(s,bool):continue
        rec.append({
            "benchmark":"AgentDojo","slug":spec["slug"],
            "family":spec["family"],"model":spec["repo"],"suite":suite,
            "utility":np.nan if not isinstance(u,bool) else int(u),
            "security":np.nan if not isinstance(s,bool) else int(s),
            "error":o.get("error"),"source_file":str(p),
        })
    return pd.DataFrame(rec)

def run_dojo(spec):
    model_file=RESULTS/f"dojo_{spec['slug']}.csv"
    slice_file=RESULTS/f"dojo_status_{spec['slug']}.csv"

    if model_file.exists() and slice_file.exists():
        old=pd.read_csv(model_file)
        old_status=pd.read_csv(slice_file)
        if len(old) and len(old_status) and old_status.status.astype(str).str.startswith("SUPPORTED").all():
            return "SUPPORTED_REUSED",old,old_status

    parts=[]
    statuses=[]
    env=os.environ.copy()
    env["OPENAI_COMPATIBLE_BASE_URL"]=LOCAL_BASE
    env["OPENAI_COMPATIBLE_API_KEY"]=LOCAL_KEY

    for suite in CFG["dojo_suites"]:
        out=RAW/"agentdojo"/spec["slug"]/suite
        out.mkdir(parents=True,exist_ok=True)
        per_slice=RESULTS/f"dojo_{spec['slug']}_{suite}.csv"

        if per_slice.exists():
            cached=pd.read_csv(per_slice)
            if len(cached):
                parts.append(cached)
                statuses.append({
                    "suite":suite,"status":"SUPPORTED_REUSED",
                    "valid":len(cached),"errors":0,"returncode":0,
                })
                continue

        cmd=[
            "uv","run","python","-m","agentdojo.scripts.benchmark",
            "--model","openai-compatible","--model-id",spec["slug"],
            "--suite",suite,"--attack","important_instructions",
            "--logdir",str(out),"--force-rerun","--max-workers","1",
        ]
        if CFG["dojo_user_tasks"] is not None:
            for x in CFG["dojo_user_tasks"]:cmd += ["--user-task",x]
        if CFG["dojo_injection_tasks"] is not None:
            for x in CFG["dojo_injection_tasks"]:cmd += ["--injection-task",x]

        p,timeout=safe_run(cmd,cwd=DOJO,env=env,timeout=1200)
        save_log(f"dojo_{spec['slug']}_{suite}.log",p,cmd)
        d=parse_dojo(out,spec,suite)
        if len(d):
            d.to_csv(per_slice,index=False)
            parts.append(d)

        valid=int(((d.utility.notna())|(d.security.notna())).sum()) if len(d) else 0
        errors=int(d.error.notna().sum()) if len(d) else 0
        st=("SUPPORTED" if valid>0 and errors==0 else
            "PARTIAL" if valid>0 else
            "TIMEOUT" if timeout else "FAILED")
        statuses.append({
            "suite":suite,"status":st,"valid":valid,
            "errors":errors,"returncode":p.returncode,
        })

    cases=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
    ss=pd.DataFrame(statuses)
    cases.to_csv(model_file,index=False)
    ss.to_csv(slice_file,index=False)
    overall=("SUPPORTED" if len(ss) and ss.status.astype(str).str.startswith("SUPPORTED").all()
             else "PARTIAL" if len(cases) else "FAILED")
    return overall,cases,ss

def parse_tau(path,spec,domain):
    try:o=json.loads(Path(path).read_text())
    except Exception:return pd.DataFrame()

    if isinstance(o,list):sims=o
    elif isinstance(o,dict):
        sims=next((o[k] for k in ["simulations","results","trajectories"]
                   if isinstance(o.get(k),list)),[])
    else:sims=[]

    rec=[]
    for i,s in enumerate(sims):
        if not isinstance(s,dict):continue
        reward=None
        if isinstance(s.get("reward_info"),dict) and isinstance(
            s["reward_info"].get("reward"),(int,float,bool)
        ):
            reward=float(s["reward_info"]["reward"])
        elif isinstance(s.get("reward"),(int,float,bool)):
            reward=float(s["reward"])
        err=s.get("error")
        if err is None and isinstance(s.get("info"),dict):err=s["info"].get("error")
        rec.append({
            "benchmark":"tau3","slug":spec["slug"],"family":spec["family"],
            "model":spec["repo"],"domain":domain,"trajectory_index":i,
            "task_id":s.get("task_id"),"reward":reward,"error":err,
        })
    return pd.DataFrame(rec)

def run_tau(spec):
    model_file=RESULTS/f"tau_{spec['slug']}.csv"
    slice_file=RESULTS/f"tau_status_{spec['slug']}.csv"

    if model_file.exists() and slice_file.exists():
        old=pd.read_csv(model_file)
        old_status=pd.read_csv(slice_file)
        if len(old) and len(old_status) and old_status.status.astype(str).str.startswith("SUPPORTED").all():
            return "SUPPORTED_REUSED",old,old_status

    parts=[]
    statuses=[]
    env=os.environ.copy()
    env["OPENAI_API_KEY"]=LOCAL_KEY
    env["OPENAI_API_BASE"]=LOCAL_BASE

    for domain in CFG["tau_domains"]:
        per_slice=RESULTS/f"tau_{spec['slug']}_{domain}.csv"
        if per_slice.exists():
            cached=pd.read_csv(per_slice)
            if len(cached) and cached.reward.notna().any():
                parts.append(cached)
                statuses.append({
                    "domain":domain,"status":"SUPPORTED_REUSED",
                    "evaluated":int(cached.reward.notna().sum()),
                    "errors":int(cached.error.notna().sum()) if "error" in cached else 0,
                    "returncode":0,
                })
                continue

        run_name=f"ntx_t4_{spec['slug']}_{domain}"
        live=TAU/"data"/"simulations"/run_name
        archive=RAW/"tau"/spec["slug"]/domain

        llm_args=json.dumps({
            "api_base":LOCAL_BASE,
            "api_key":LOCAL_KEY,
            "temperature":0.0,
            "max_tokens":768,
        })

        cmd=[
            "uv","run","tau2","run",
            "--domain",domain,
            "--agent-llm","openai/"+spec["slug"],
            "--user-llm","openai/"+spec["slug"],
            "--agent-llm-args",llm_args,
            "--user-llm-args",llm_args,
            "--num-trials","1",
            "--task-split-name","base",
            "--max-steps",str(CFG["tau_steps"]),
            "--max-errors","3",
            "--max-concurrency","1",
            "--max-retries","1",
            "--retry-delay","2",
            "--seed",str(SEED),
            "--save-to",run_name,
            "--auto-resume","--verbose-logs","--llm-log-mode","latest",
        ]
        if CFG["tau_tasks"] is not None:cmd += ["--num-tasks",str(CFG["tau_tasks"])]

        p,timeout=safe_run(cmd,cwd=TAU,env=env,timeout=1500)
        save_log(f"tau_{spec['slug']}_{domain}.log",p,cmd)

        if live.exists():
            if archive.exists():shutil.rmtree(archive)
            shutil.copytree(live,archive)

        d=(parse_tau(archive/"results.json",spec,domain)
           if (archive/"results.json").exists() else pd.DataFrame())
        if len(d):
            d.to_csv(per_slice,index=False)
            parts.append(d)

        n=int(d.reward.notna().sum()) if len(d) else 0
        errors=int(d.error.notna().sum()) if len(d) else 0
        st=("SUPPORTED" if n>0 and errors==0 else
            "PARTIAL" if n>0 else
            "TIMEOUT" if timeout else "FAILED")
        statuses.append({
            "domain":domain,"status":st,"evaluated":n,
            "errors":errors,"returncode":p.returncode,
        })

    cases=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
    ss=pd.DataFrame(statuses)
    cases.to_csv(model_file,index=False)
    ss.to_csv(slice_file,index=False)
    overall=("SUPPORTED" if len(ss) and ss.status.astype(str).str.startswith("SUPPORTED").all()
             else "PARTIAL" if len(cases) else "FAILED")
    return overall,cases,ss

print("✅ Resumable benchmark runners ready.")


In [ ]:
# CELL 7 — RUN-ONE-MODEL HELPER (T4 RESUMABLE)

def get_spec(slug):
    for s in MODEL_REGISTRY:
        if s["slug"]==slug:return s
    raise KeyError(slug)

def mark_model(slug,**updates):
    global STATE
    STATE.setdefault(slug,{})
    STATE[slug].update(updates)
    STATE[slug]["updated_utc"]=datetime.now(timezone.utc).isoformat()
    save_checkpoint(STATE)

def run_one_model(slug):
    spec=get_spec(slug)
    print("\n"+"="*96)
    print("T4 MODEL:",spec["repo"],"| family:",spec["family"])
    print("="*96)

    if not spec["gpu_eligible"]:
        mark_model(slug,status="SKIPPED_VRAM")
        print("SKIP: configured VRAM threshold exceeds this T4.")
        return
    if not spec["token_eligible"]:
        mark_model(slug,status="SKIPPED_GATED")
        print("SKIP: gated model. Set HF_TOKEN only if you already have access.")
        return

    prev=STATE.get(slug,{})
    if prev.get("status")=="COMPLETE":
        print("✅ Already complete according to checkpoint.")
        return

    local_path=None
    try:
        mark_model(slug,status="DOWNLOADING")
        local_path,revision,size_gb=download_model(spec)
        mark_model(slug,status="DOWNLOADED",revision=revision,remote_size_gb=round(size_gb,3))

        mark_model(slug,status="STARTING_SERVER")
        client=start_server(spec,local_path)
        mark_model(slug,status="SERVER_READY")

        pf=tool_preflight(client,spec)
        (RESULTS/f"preflight_{slug}.json").write_text(json.dumps(pf,indent=2))
        mark_model(
            slug,status="PREFLIGHT_DONE",
            chat_ok=pf["chat_ok"],auto_tool_ok=pf["auto_tool_ok"],
            forced_tool_ok=pf["forced_tool_ok"],preflight_error=pf["error"],
        )

        if not (pf["chat_ok"] and pf["auto_tool_ok"]):
            mark_model(slug,status="SKIPPED_TOOL_PREFLIGHT")
            print("SKIP: autonomous tool-call preflight failed.")
            return

        bf_st,bf_df,bf_rc=run_bfcl(spec)
        mark_model(slug,bfcl=bf_st,bfcl_returncode=bf_rc)
        print("BFCL:",bf_st,"rows:",len(bf_df))

        run_all=spec["full_bench"] or MODE=="FULL"
        if run_all:
            dj_st,dj_df,dj_ss=run_dojo(spec)
            mark_model(slug,agentdojo=dj_st)
            print("AgentDojo:",dj_st,"cases:",len(dj_df))

            tau_st,tau_df,tau_ss=run_tau(spec)
            mark_model(slug,tau3=tau_st)
            print("tau3:",tau_st,"trajectories:",len(tau_df))
        else:
            dj_st="SUPPLEMENTARY_NOT_RUN"
            tau_st="SUPPLEMENTARY_NOT_RUN"
            mark_model(slug,agentdojo=dj_st,tau3=tau_st)

        if run_all:
            complete=(
                str(bf_st).startswith("SUPPORTED") and
                str(dj_st).startswith("SUPPORTED") and
                str(tau_st).startswith("SUPPORTED")
            )
        else:
            complete=str(bf_st).startswith("SUPPORTED")

        mark_model(slug,status="COMPLETE" if complete else "PARTIAL_COMPLETE")

    except Exception as e:
        mark_model(slug,status="ERROR",error=repr(e))
        print("MODEL ERROR:",repr(e))
    finally:
        stop_server()
        if local_path is not None and not KEEP_MODEL_WEIGHTS:
            shutil.rmtree(local_path,ignore_errors=True)
        gc.collect()

    print("Checkpoint:",STATE.get(slug,{}))

print("✅ run_one_model() ready.")


In [ ]:
# CELL 7A — granite31_2b
run_one_model("granite31_2b")

In [ ]:
# CELL 7B — qwen25_3b
run_one_model("qwen25_3b")

In [ ]:
# CELL 7C — phi4_mini
run_one_model("phi4_mini")

In [ ]:
# CELL 7D — ministral3_3b
run_one_model("ministral3_3b")

In [ ]:
# CELL 7E — xlam_1b
run_one_model("xlam_1b")

In [ ]:
# CELL 7F — xlam_3b
run_one_model("xlam_3b")

In [ ]:
# CELL 7G — qwen25_7b_awq
run_one_model("qwen25_7b_awq")

In [ ]:
# CELL 7H — llama32_3b
run_one_model("llama32_3b")

In [ ]:
# CELL 8 — RELOAD PERSISTED RESULTS + PAPER CLOSURE GATE

STATE=load_checkpoint()
status_rows=[]
bfcl_parts=[]
dojo_parts=[]
tau_parts=[]

for spec in MODEL_REGISTRY:
    s=STATE.get(spec["slug"],{})
    status_rows.append({
        "slug":spec["slug"],"family":spec["family"],"repo":spec["repo"],
        "counts_for_closure":spec["counts_for_closure"],
        "full_bench":spec["full_bench"],**s,
    })

    p=RESULTS/f"bfcl_{spec['slug']}.csv"
    if p.exists():
        try:
            d=pd.read_csv(p)
            if len(d):bfcl_parts.append(d)
        except Exception:pass

    p=RESULTS/f"dojo_{spec['slug']}.csv"
    if p.exists():
        try:
            d=pd.read_csv(p)
            if len(d):dojo_parts.append(d)
        except Exception:pass

    p=RESULTS/f"tau_{spec['slug']}.csv"
    if p.exists():
        try:
            d=pd.read_csv(p)
            if len(d):tau_parts.append(d)
        except Exception:pass

status_df=pd.DataFrame(status_rows)
bfcl_df=pd.concat(bfcl_parts,ignore_index=True) if bfcl_parts else pd.DataFrame()
dojo_df=pd.concat(dojo_parts,ignore_index=True) if dojo_parts else pd.DataFrame()
tau_df=pd.concat(tau_parts,ignore_index=True) if tau_parts else pd.DataFrame()

status_df.to_csv(RESULTS/"10_model_run_status.csv",index=False)
bfcl_df.to_csv(RESULTS/"20_bfcl_native_metrics.csv",index=False)
dojo_df.to_csv(RESULTS/"21_agentdojo_native_cases.csv",index=False)
tau_df.to_csv(RESULTS/"22_tau3_native_cases.csv",index=False)

display(status_df[[c for c in [
    "slug","family","status","bfcl","agentdojo","tau3","chat_ok","auto_tool_ok","error"
] if c in status_df.columns]])

summary_rows=[]
if len(bfcl_df):
    for (slug,family,metric),g in bfcl_df.groupby(["slug","family","metric"]):
        summary_rows.append({
            "benchmark":"BFCL-v4","slug":slug,"family":family,
            "metric":metric,"n":len(g),"mean":float(g.score.mean()),
        })
if len(dojo_df):
    for (slug,family),g in dojo_df.groupby(["slug","family"]):
        if g.utility.notna().any():
            summary_rows.append({
                "benchmark":"AgentDojo","slug":slug,"family":family,
                "metric":"utility","n":int(g.utility.notna().sum()),"mean":float(g.utility.mean()),
            })
        if g.security.notna().any():
            summary_rows.append({
                "benchmark":"AgentDojo","slug":slug,"family":family,
                "metric":"security","n":int(g.security.notna().sum()),"mean":float(g.security.mean()),
            })
if len(tau_df):
    for (slug,family),g in tau_df.groupby(["slug","family"]):
        summary_rows.append({
            "benchmark":"tau3","slug":slug,"family":family,
            "metric":"reward","n":int(g.reward.notna().sum()),"mean":float(g.reward.mean()),
        })

summary=pd.DataFrame(summary_rows)
summary.to_csv(RESULTS/"30_external_model_summary.csv",index=False)
display(summary)

complete_families=[]
closure_df=status_df[status_df["counts_for_closure"]==True]
for family,g in closure_df.groupby("family"):
    ok=False
    for _,r in g.iterrows():
        if (
            str(r.get("bfcl","")).startswith("SUPPORTED") and
            str(r.get("agentdojo","")).startswith("SUPPORTED") and
            str(r.get("tau3","")).startswith("SUPPORTED")
        ):
            ok=True
            break
    if ok:complete_families.append(family)

complete_families=sorted(set(complete_families))
closure_supported=len(complete_families)>=3

claims=pd.DataFrame([
    {"claim":"Local BFCL-v4 evidence","status":"SUPPORTED" if len(bfcl_df) else "MISSING"},
    {"claim":"Local AgentDojo evidence","status":"SUPPORTED" if len(dojo_df) else "MISSING"},
    {"claim":"Local tau3 evidence","status":"SUPPORTED" if len(tau_df) else "MISSING"},
    {
        "claim":"At least 3 independent T4 model families complete all required benchmarks",
        "status":"SUPPORTED" if closure_supported else "INCOMPLETE",
    },
    {
        "claim":"T4 local external paper-closure gate",
        "status":"SUPPORTED" if closure_supported else "INCOMPLETE",
    },
])
claims.to_csv(RESULTS/"31_claim_gate.csv",index=False)
display(claims)
print("Complete independent families:",complete_families)


In [ ]:
# CELL 9 — T4 PAPER TABLES / PROVENANCE — PAPER TABLES / FIGURES / PROVENANCE

summary.to_latex(
    PAPER/"local_external_summary.tex",
    index=False,float_format="%.4f"
)
claims.to_latex(
    PAPER/"local_external_claim_gate.tex",
    index=False
)
status_df.to_csv(PAPER/"local_model_status.csv",index=False)

if len(summary):
    for (benchmark,metric),g in summary.groupby(["benchmark","metric"]):
        gg=g.sort_values("mean")
        fig,ax=plt.subplots(figsize=(8,max(3,0.45*len(gg)+1)))
        ax.barh(gg["slug"],gg["mean"])
        ax.set_title(f"{benchmark}: {metric}")
        ax.set_xlabel(metric)
        fig.tight_layout()
        safe=re.sub(r"[^A-Za-z0-9]+","_",f"{benchmark}_{metric}")
        fig.savefig(PAPER/f"{safe}.png",dpi=220,bbox_inches="tight")
        plt.show()

if closure_supported:
    section=(
        "\\paragraph{External local-model validation.}\n"
        "We additionally evaluated the runtime using locally served open-weight models "
        "without relying on commercial inference APIs. At least three independent model "
        f"families completed BFCL-v4, AgentDojo, and $\\tau^3$: {', '.join(complete_families)}. "
        "Each model was served independently through a local OpenAI-compatible vLLM endpoint, "
        "and only native benchmark outputs with evaluated cases were retained."
    )
else:
    section=(
        "\\paragraph{External local-model validation.}\n"
        "We evaluated a set of locally served open-weight models on BFCL-v4, AgentDojo, "
        "and $\\tau^3$. Because fewer than three independent model families completed all "
        "three benchmark families, these results are reported as bounded external evidence "
        "rather than a complete cross-family generalization claim."
    )

(PAPER/"local_external_validation_section.tex").write_text(section+"\n")

# Model/revision provenance.
manifest={
    "experiment":"NTX-T4-LOCAL-OPEN-WEIGHT-EXTERNAL-CLOSURE",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "mode":MODE,
    "gpu_name":GPU_NAME,
    "gpu_vram_gb":VRAM_GB,
    "benchmark_versions":BENCHMARK_VERSIONS,
    "complete_families":complete_families,
    "closure_supported":closure_supported,
    "model_status":status_df.to_dict("records"),
    "config":CFG,
}
(RESULTS/"FINAL_MANIFEST.json").write_text(json.dumps(manifest,indent=2,default=str))

# Hash research outputs (not giant model weights).
hash_rows=[]
for label,folder in [("results",RESULTS),("paper",PAPER),("logs",LOGS)]:
    for p in sorted(folder.rglob("*")):
        if p.is_file() and p.name!="SHA256_MANIFEST.csv":
            h=hashlib.sha256()
            with open(p,"rb") as f:
                for chunk in iter(lambda:f.read(1024*1024),b""):h.update(chunk)
            hash_rows.append({"file":f"{label}/{p.relative_to(folder)}","sha256":h.hexdigest()})
pd.DataFrame(hash_rows).to_csv(RESULTS/"SHA256_MANIFEST.csv",index=False)

In [ ]:
# CELL 10 — T4 FINAL ZIP / DOWNLOAD — ZIP ALL RESULTS AND DOWNLOAD

stage=BASE/"final_package"
if stage.exists():shutil.rmtree(stage)
stage.mkdir()

for src,name in [
    (RESULTS,"results"),
    (RAW,"raw_benchmark_outputs"),
    (LOGS,"logs"),
    (PAPER,"paper_integration"),
]:
    if src.exists():
        shutil.copytree(src,stage/name)

# Small reproducibility files only; do not package multi-GB model weights.
(stage/"README.txt").write_text(
    "NiyamTrace-X local open-weight external validation package.\n"
    "Model weights are intentionally excluded. See results/FINAL_MANIFEST.json "
    "and results/10_model_run_status.csv for model IDs/revisions/status.\n"
)

ZIP=ARCH/"NTX_T4_LOCAL_EXTERNAL_CLOSURE_RESULTS.zip"
if ZIP.exists():ZIP.unlink()

with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
    for p in stage.rglob("*"):
        if p.is_file():
            z.write(p,arcname=str(p.relative_to(stage)))

with zipfile.ZipFile(ZIP) as z:
    bad=z.testzip()

h=hashlib.sha256()
with open(ZIP,"rb") as f:
    for chunk in iter(lambda:f.read(1024*1024),b""):h.update(chunk)

print("Closure gate:", "SUPPORTED" if closure_supported else "INCOMPLETE")
print("Complete families:",complete_families)
print("ZIP integrity:", "PASS" if bad is None else bad)
print("ZIP:",ZIP)
print("SHA256:",h.hexdigest())

try:
    from google.colab import files
    files.download(str(ZIP))
except Exception as e:
    print("Auto-download unavailable:",repr(e))